# 02_Calidad y limpieza

In [ ]:
import pandas as pd
import numpy as np

# 1. CARGA DEL DATASET DESDE URL
url = "/content/streaming_users_dirty.json"
df_raw = pd.read_json(url)
df_raw.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [ ]:
# 2. PRESERVAR ORIGINAL Y CONFIGURAR AUDITORÍA
df = df_raw.copy()
log  = []

def registrar(df, paso, desc):
  log.append({
      'Paso' : paso,
      'Descripción' : desc,
      'Filas' : len(df),
      'Nulos' : df.isnull().sum().sum(),
      'Retención (%)' : round(len(df) / 8160*100, 2 )
  })

registrar(df, 0, "Dataset original")

Luego de cargar el dataset desde url, se genera una copia para trabajar de forma aislada, permitiendo reiniciar el pipeline desde cero sin necesidad de reordenar o volver a cargar los archivos en caso de error. El pipeline automatiza de forma secuencial el flujo de los datos sobre su estado bruto (raw) hasta un estado limpio y óptimo, asegurando consistencia y reproducibilidad en los resultados.

Paso 1 : Eliminación de duplicados

In [ ]:
df.columns.difference(['age'])

Index(['country', 'customer_support_tickets', 'favorite_genre',
       'last_login_date', 'monthly_watch_time_mins', 'subscription_plan',
       'user_id'],
      dtype='object')

Con la función "df.columns.difference(['age'])" realizamos un filtrado por subconjunto, considerando todas las columnas menos la edad, para garantizar quelas identidades repetidas sean unificadas correctamente, limpiando el ruido de redundancia sin alterar el comportamiento de consumo real de la plataforma.

In [ ]:
df.drop_duplicates(subset=df.columns.difference(['age'])).reset_index(drop=False)

,index,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1
...,...,...,...,...,...,...,...,...,...
8027,8146,11796,21,Premium,1427.8,Perú,Drama,2022-04-24,2
8028,8147,13802,33,Básico,482.4,Perú,Comedia,2022-08-29,1
8029,8154,11586,17,Básico,1123.6,Brasil,Crime,2019-03-20,1
8030,8157,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0


El dataset original presentaba un índice máximo de 8158 registros, los cuales quedaron en 8031 registros después de eliminar los duplicados (indexados de 0 a 8030). Este proceso eliminó con éxito filas redundantes que compartían perfiles idénticos de consumo, plan de subscripción, país, limpiando la base de datos sin generar pérdidas masivas de información en esta fase inicial del pipeline. La columna "index" contiene el orden viejo de los registros, lo que permite determinar qué posición tenía originalmente cada usuario antes de limpiar el dataset.

In [ ]:
# 3. PASO 1: ELIMINACIÓN DE DUPLICADOS
n_antes = len(df)
df = df.drop_duplicates(
    subset = df.columns.difference(['age'])
).reset_index(drop=True)

print(f"Paso 1 - Duplicados eliminados: {n_antes - len(df)}")

registrar(df, 1, "Eliminación de duplicados")

Paso 1 - Duplicados eliminados: 128


Una vez que eliminamos los registros duplicados, utilizamos "reset_index(drop=True) para descartar el índice viejo porque ya no aporta valor analítico.

Paso 2: Normalización de variables categóricas

In [ ]:
# 4. Paso 2: NORMALIZACIÓN DE CATEGÓRICAS

# Listamos las variables categóricas a normalizar
variables_categoricas = ['subscription_plan', 'country', 'favorite_genre']

print("\nPaso 2 - Valores únicos ANTES:")
for col in variables_categoricas:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

# --- ESTRUCTURA DE DICCIONARIOS ---

mapa_plan = {
    "premium": "Premium",
    "premiun": "Premium",
    "básico": "Básico",
    "basico": "Básico",
    "basic": "Básico",
    "estándar": "Estándar",
    "estandar": "Estándar",
    "standard": "Estándar",
    "std": "Estándar"
}

mapa_pais = {
    "arg": "Argentina", "argentina": "Argentina",
    "bra": "Brasil", "brasil": "Brasil", "brazil": "Brasil",
    "col": "Colombia", "colombia": "Colombia",
    "chl": "Chile", "chile": "Chile",
    "mex": "México", "méxico": "México", "mexico": "México",
    "per": "Perú", "perú": "Perú", "peru": "Perú",
    "ury": "Uruguay", "uruguay": "Uruguay"
}

mapa_genero = {
    "crime": "Crimen", "crimen": "Crimen",
    "acción": "Acción", "accion": "Acción", "action": "Acción",
    "thriller": "Thriller", "thriller ": "Thriller", " thriller": "Thriller",
    "horror": "Terror", "terror": "Terror",
    "adventure": "Aventura", "aventura": "Aventura",
    "drama": "Drama", "romance": "Romance",
    "comedy": "Comedia", "comedia": "Comedia",
    "documentary": "Documental", "doc": "Documental", "documental": "Documental",
    "nan": None, "none": None  # Mapeo prolijo para forzar nulos reales
}

# --- APLICACIÓN DE MAPEOS  ---
# Usamos .str.strip().str.lower() antes para asegurarnos de que coincidan con las llaves del diccionario
df['subscription_plan'] = df['subscription_plan'].astype(str).str.strip().str.lower().map(mapa_plan)
df['country'] = df['country'].astype(str).str.strip().str.lower().map(mapa_pais)
df['favorite_genre'] = df['favorite_genre'].astype(str).str.strip().str.lower().map(mapa_genero)

print("\nPaso 2 - Valores únicos DESPUÉS:")
for col in variables_categoricas:
    # Agregamos una conversión a string temporal en el print por si quedaron NaN reales
    print(f"{col}: {sorted(df[col].dropna().unique())}")

registrar(df, 2, "Normalización categorías")


Paso 2 - Valores únicos ANTES:
subscription_plan: ['BASICO', 'Basic', 'Básico', 'Estándar', 'Estándar ', 'PREMIUM', 'Premium', 'Premium ', 'Premiun', 'STANDARD', 'Std', 'basico', 'básico', 'estandar', 'premium']
country: ['ARG', 'Argentina', 'Argentina ', 'BRA', 'Brasil', 'Brazil', 'CHL', 'COL', 'Chile', 'Chile ', 'Colombia', 'MEX', 'Mexico', 'México', 'PER', 'Peru', 'Perú', 'URY', 'Uruguay', 'argentina', 'brasil', 'chile', 'colombia', 'méxico', 'perú', 'uruguay']
favorite_genre: ['ACCIÓN', 'Acción', 'Action', 'COMEDIA', 'CRIME', 'Comedia', 'Comedia ', 'Crime', 'Crimen', 'DOC', 'DRAMA', 'Documental', 'Documentary', 'Drama', 'Drama ', 'ROMANCE', 'Romance', 'Romance ', 'THRILLER', 'Thriller', 'Thriller ', 'accion', 'comedy', 'crime', 'documental', 'drama', 'romance', 'thriler']

Paso 2 - Valores únicos DESPUÉS:
subscription_plan: ['Básico', 'Estándar', 'Premium']
country: ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']
favorite_genre: ['Acción', 'Comedia', 'Cri

La normalización de variables categóricas "subscription_plan", "country" y "favorite_genre" permite pasar de strings inconsistentes, como errores de tipeo, mezcla de idiomas y abreviaciones, hacia valores estandarizados y limpios. Para garantizar la consistencia, utilizamos la técnica de mapeo explícito mediante diccionarios estructuradis y la función .map().

Paso 3: Tratamiento de valores imposibles

In [ ]:
# 5. PASO 3: TRATAMIENTO DE VALORES IMPOSIBLES

# PASO 3a. EDAD (age): Rango clínico/lógico válido 12 - 100
age_invalido = df[(df['age'] < 12) | (df['age'] > 100)]
print(f"Paso 3a - Edades inválidas detectadas: {len(age_invalido)}")
print(age_invalido[['user_id', 'age', 'subscription_plan', 'favorite_genre']].to_string(index=False))

df.loc[(df['age'] < 12) | (df['age'] > 100), 'age'] = np.nan

# PASO 3b. MINUTOS DE REPRODUCCIÓN: Valores imposibles
watch_invalido = df[df['monthly_watch_time_mins'] < 0]
print(f"\nPaso 3b - Minutos imposibles (negativos) detectados: {len(watch_invalido)}")
print(watch_invalido[['user_id', 'monthly_watch_time_mins', 'customer_support_tickets']].to_string(index=False))

df.loc[df['monthly_watch_time_mins'] < 0, 'monthly_watch_time_mins'] = np.nan

# PASO 3c. TICKETS DE SOPORTE: Valores imposibles
tickets_invalido = df[df['customer_support_tickets'] < 0]
print(f"\nPaso 3c - Tickets imposibles (negativos) detectados: {len(tickets_invalido)}")
print(tickets_invalido[['user_id', 'monthly_watch_time_mins', 'customer_support_tickets']].to_string(index=False))

df.loc[df['customer_support_tickets'] < 0, 'customer_support_tickets'] = np.nan

# PASO 3d. CONVERSION DE FECHAS (last_login_date)
df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')


# --- RESUMEN DE CONTROL DE NULOS ---
print("\n--- RESUMEN DE CONTROL DE NULOS EN PASO 3 ---")
print(f"Nulos en Edad:    {df['age'].isnull().sum()}")
print(f"Nulos en Minutos: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"Nulos en Tickets: {df['customer_support_tickets'].isnull().sum()}")
print(f"Nulos en Fechas:  {df['last_login_date'].isnull().sum()}")

registrar(df, 3, "Corrección valores imposibles en edad, minutos, tickets y fechas")

Paso 3a - Edades inválidas detectadas: 120
 user_id  age subscription_plan favorite_genre
   10049    0            Básico        Romance
   10194  130            Básico          Drama
   10310    0          Estándar         Crimen
   10324  130            Básico       Thriller
   10426  150           Premium         Acción
   10442   -5          Estándar         Crimen
   10529  130            Básico       Thriller
   10573   -5           Premium     Documental
   10640  150           Premium          Drama
   10655  130           Premium     Documental
   10751  150            Básico         Acción
   11028  150            Básico         Acción
   11099  150            Básico        Romance
   11126    0           Premium     Documental
   11159    0           Premium     Documental
   11242   -5          Estándar       Thriller
   11344    4          Estándar        Comedia
   11374  130            Básico         Acción
   11422    0          Estándar        Comedia
   11445  150    

Hay variables lógicas y/o biológicas que definen un rango válido, como la edad, que en este caso se puede establecer entre 12 y 100. Cualquier registro fuera de ese rango, se clasifica como inválido; las variables cuantitativas "momthly_watch_time_mins" y "customer_support_tickets" no puede tomar valores menores a cero, por lo que los valores negativos detectados representan errores en el sistema de registro. Asimismo, las fechas deben ser transformadas a tipo fecha para poder realizar métricas; en este caso se incorpora el errors='coerce' para que cualquier string corrupto, mal formateado o imposible se fuerce automaticamente a tomar un valor nulo, para ser tratado con posterioridad a través de otro método. Para todas las variables numéricas, los casos inválidos se reemplazan por np.nan utilizando la indexación .loc, lo que evita eliminar prematuramente filas completas, preservando los datos válidos que ese mismo usuario pueda tener en otras columnas, para imputarlas en fases posteriores del pipeline.

In [ ]:
# 6. PASO 4: DIAGNÓSTICO DEL MECANISMO DE FALTA

# 4A. ¿La falta de 'age' depende del consumo de minutos? (MNAR)
# Creamos la variable indicadora artificial (1 si falta edad, 0 si está)
df['age_falta'] = df['age'].isnull().astype(int)

# Contrastamos el consumo promedio de minutos entre ambos grupos
# Usamos .dropna() temporalmente en minutos para que no altere el promedio general
minutos_con_age = df[df['age_falta'] == 0]['monthly_watch_time_mins'].mean()
minutos_sin_age = df[df['age_falta'] == 1]['monthly_watch_time_mins'].mean()

print("--- Diagnóstico MNAR: variable 'age' ---")
print(f"Minutos promedio CON age registrada : {minutos_con_age:.2f}")
print(f"Minutos promedio SIN age registrada : {minutos_sin_age:.2f}")
print(f"Diferencia de consumo:  : {minutos_sin_age - minutos_con_age:.2f}")

# 4B. ¿La falta de 'age' depende del plan de suscripción?
plan_tasa_falta = df.groupby('subscription_plan')['age_falta'].mean() * 100

print("\nTasa de falta en 'age' por tipo de plan:")
print(plan_tasa_falta.round(2))


# 4C. ¿La falta de 'monthly_watch_time_mins' depende del país? (MAR)
df['watch_falta'] = df['monthly_watch_time_mins'].isnull().astype(int)
tasa_falta_pais = df.groupby('country')['watch_falta'].mean() * 100

print("\n--- Diagnóstico MAR: variable 'monthly_watch_time_mins' ---")
print("Tasa de falta de minutos por país:")
print(tasa_falta_pais.round(2).sort_values(ascending=False))

# 4D. ¿La falta de 'favorite_genre' depende del Plan de Suscripción?
df['genre_falta'] = df['favorite_genre'].isnull().astype(int)
tasa_falta_plan_genre = df.groupby('subscription_plan')['genre_falta'].mean() * 100

print("\n--- Diagnóstico MAR: variable 'favorite_genre' ---")
print("Tasa de falta de género favorito por tipo de plan:")
print(tasa_falta_plan_genre.round(2))

--- Diagnóstico MNAR: variable 'age' ---
Minutos promedio CON age registrada : 1124.17
Minutos promedio SIN age registrada : 801.70
Diferencia de consumo:  : -322.48

Tasa de falta en 'age' por tipo de plan:
subscription_plan
Básico      1.47
Estándar    1.69
Premium     1.19
Name: age_falta, dtype: float64

--- Diagnóstico MAR: variable 'monthly_watch_time_mins' ---
Tasa de falta de minutos por país:
country
Uruguay      3.75
México       3.18
Perú         3.16
Chile        3.00
Brasil       2.92
Colombia     2.53
Argentina    2.52
Name: watch_falta, dtype: float64

--- Diagnóstico MAR: variable 'favorite_genre' ---
Tasa de falta de género favorito por tipo de plan:
subscription_plan
Básico      3.16
Estándar    3.18
Premium     2.64
Name: genre_falta, dtype: float64


Los valores faltantes (nulos) no se imputan con una media o mediana general de forma automática, sino que primero se debe realizar un diagnóstico estadístico para determinar qué mecanismos utilizaremos, garantizando que la imputación no inserte sesgos que alteren la dustribución real de los usuarios en la plataforma de streaming. Así, realizamos el diagnóstico por mecanismo de falta, que nos permitió confirmar la presencia de un sesgo MNAR en la variable 'age' debido a que la edad se oculta de forma intencional por diferentes motivos en una plataforma y esto genera sesgos, es decir, la causa de la ausencia del dato está directamente relacionada con el valor del dato que falta o con el comportamiento del usuario. De esta manera, se puede determinar que el grupo que oculta su edad tiene un comportamiento de reproducción diferente, con una diferencia de más de 320 minutos, en comparación con quienes sí la informan.

In [ ]:
# 7. PASO 5: WINSORIZACIÓN ROBUSTA DE OUTLIERS (MÉTODO IQR)

# --- 5a. Winsorización Robusta en Minutos de Reproducción ---
Q1_mins = df['monthly_watch_time_mins'].quantile(0.25)
Q3_mins = df['monthly_watch_time_mins'].quantile(0.75)
IQR_mins = Q3_mins - Q1_mins

# Usamos el umbral de Tukey para outliers severos (Q3 + 3 * IQR)
limite_sup_mins = Q3_mins + (3 * IQR_mins)

outliers_mins = df[df['monthly_watch_time_mins'] > limite_sup_mins]
print("--- 5a. Winsorización Robusta (IQR): 'monthly_watch_time_mins' ---")
print(f"Límite superior calculado (Tukey): {limite_sup_mins:.2f} minutos")
print(f"Cantidad de outliers a recortar: {len(outliers_mins)}")

# Aplicamos el recorte (clip) superior
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].clip(upper=limite_sup_mins)


# --- 5b. Winsorización Robusta en Tickets de Soporte ---
Q1_tickets = df['customer_support_tickets'].quantile(0.25)
Q3_tickets = df['customer_support_tickets'].quantile(0.75)
IQR_tickets = Q3_tickets - Q1_tickets

# Umbral de Tukey para outliers severos (Q3 + 3 * IQR)
limite_sup_tickets = Q3_tickets + (3 * IQR_tickets)

outliers_tickets = df[df['customer_support_tickets'] > limite_sup_tickets]
print("\n--- 5b. Winsorización Robusta (IQR): 'customer_support_tickets' ---")
print(f"Límite superior calculado (Tukey): {limite_sup_tickets:.2f} tickets")
print(f"Cantidad de outliers a recortar: {len(outliers_tickets)}")

# Aplicamos el recorte (clip) superior
df['customer_support_tickets'] = df['customer_support_tickets'].clip(upper=limite_sup_tickets)


# --- REGISTRO DEL PASO EN EL PIPELINE ---
registrar(df, 5, "Winsorización robusta IQR en minutos y tickets")

--- 5a. Winsorización Robusta (IQR): 'monthly_watch_time_mins' ---
Límite superior calculado (Tukey): 2705.40 minutos
Cantidad de outliers a recortar: 139

--- 5b. Winsorización Robusta (IQR): 'customer_support_tickets' ---
Límite superior calculado (Tukey): 4.00 tickets
Cantidad de outliers a recortar: 81


Para el tratamiento de outliers en "monthly_watch_time_mins" y "customer_support_tickets" implementamos un algoritmo de winsorización robusta basado en Rango Intercuartílico (IQR) y el Umbral de Tukey para outliers severos. En lugar de eliminar las filas y reducir la muestra, utilizamos la función .clip(upper=) para recortar y nivelar los valores extremos superiores fijando una frontera estadística basada en los percentiles Q1 y Q3. El enfoque convencional K=3 asume que los datos siguen una distribución normal y simétrica. Sin embargo, variables operativas de compoprtamiento digital, como minutos de reproducción y tickets de soporte, suelen presentar asimetrías severas con largas colas a la derecha y errores de carga masivos. Al usar la media y el desvío estándar, esos pocos valores extremos inflan los indicadores, estirando los límites a rangos lógicos para el negocio. Cuando aplicamos k=3 el límite superior en "monthly_watch_time_mins" se había disparado a 17,222.13 minutos, lo que equvalía a casi 10 horas de streaming continuas por día, por el sesgo bruto de 99999 que distrosiona la media. En "customer_support_tickets", el límite superior se había estirado artificialmente hasta los 36.16 tickets, pero generaba demasiado ruido porque la mayoría de los usuarios acumulaba entre 0 y 5. Al cambiar los mecanismos, nos permitió establecer fronteras de corte coherente y alineados con la realidad operativa de la plataforma, bajando los umbrales a 2705.40 minutos y 4 tickets.

In [ ]:
# 8. PASO 6: IMPUTACIÓN DIFERENCIADA POR MECANISMO DE FALTA

# --- 1. EDAD (Mecanismo: MNAR) ---
df['age'] = df.groupby(['favorite_genre', 'subscription_plan'])['age'].transform(lambda x: x.fillna(x.median()))
df['age'] = df['age'].fillna(df['age'].median())

# --- 2. MINUTOS DE REPRODUCCIÓN (Mecanismo: MAR) ---
df['monthly_watch_time_mins'] = df.groupby(['country', 'subscription_plan'])['monthly_watch_time_mins'].transform(lambda x: x.fillna(x.median()))


# --- 3. TICKETS DE SOPORTE (Mecanismo: MCAR/MAR) ---
df['customer_support_tickets'] = df.groupby('subscription_plan')['customer_support_tickets'].transform(lambda x: x.fillna(x.median())).round().astype(int)


# --- 4. FECHAS (Mecanismo: MAR) ---
df['last_login_date'] = df.groupby('country')['last_login_date'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else pd.NaT))

# --- 5. GÉNERO FAVORITO (Variable Categórica)
df['favorite_genre'] = df.groupby('subscription_plan')['favorite_genre'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Drama"))

# --- RESUMEN DE CONTROL DE NULOS FINAL ---
print("RECUENTO DE NULOS POST-IMPUTACIÓN")
print("-" * 40)
print(df[["age", "monthly_watch_time_mins", "customer_support_tickets", "last_login_date", "favorite_genre"]].isnull().sum())

registrar(df, 6, "Imputación diferenciada por mecanismo")

RECUENTO DE NULOS POST-IMPUTACIÓN
----------------------------------------
age                         0
monthly_watch_time_mins     0
customer_support_tickets    0
last_login_date             0
favorite_genre              0
dtype: int64


Una vez identificados los comportamientos y sesgos de las variables con nulos en el paso anterior, se rechaza formalmente el uso de imputaciones globales, como rellenar a todos los usuarios con un único promedio general y se diseña una arquitectura de imputación diferenciada, aplicando transformaciones agrupadas mediante .groupby() y segmentando las soluciones según la naturaleza de cada variable. Así, agrupamos el dataset por el género de contenido preferido y el nivel de subscripción, lo que permite capturar el perfil de usuario implícito, mitigando el sesgo original. Para el tiempo de consumo y la última fecha de inicio de sesión, agrupamos los datos combinando los factores de localización y tipo de plan. Con respecto a las variables categóricas de "género favorito" implementamos una función lambda condicionada para imputar la variable cualitativa utilizando la moda local de cada plan de subscripción, incorporando una excepción estricta para asignar "drama" como valor por defecto en caso de que un segmento específico se encuentre completamente vacío. En este caso, no es matemáticamente posible aplicar medias ni medianas. Por lo tanto, usar la moda segmentada permite que el dato flotante adopte la preferencia más común y representativa de los usuarios que comparten su mismo nivel de facturación.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8032 entries, 0 to 8031
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   8032 non-null   int64         
 1   age                       8032 non-null   float64       
 2   subscription_plan         8032 non-null   object        
 3   monthly_watch_time_mins   8032 non-null   float64       
 4   country                   8032 non-null   object        
 5   favorite_genre            8032 non-null   object        
 6   last_login_date           8032 non-null   datetime64[ns]
 7   customer_support_tickets  8032 non-null   int64         
 8   age_falta                 8032 non-null   int64         
 9   watch_falta               8032 non-null   int64         
 10  genre_falta               8032 non-null   int64         
dtypes: datetime64[ns](1), float64(2), int64(5), object(3)
memory usage: 690.4+ KB


In [ ]:
# --- CORRECCIÓN DE TIPOS DE DATOS PARA EL EDA ---

# 1. Convertimos el ID de usuario a tipo objeto (String)
df['user_id'] = df['user_id'].astype(str)

# 2. Convertimos la edad a tipo entero (Integer)
df['age'] = df['age'].astype('int64')

# Verificamos que los cambios se hayan aplicado correctamente
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8032 entries, 0 to 8031
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   8032 non-null   object        
 1   age                       8032 non-null   int64         
 2   subscription_plan         8032 non-null   object        
 3   monthly_watch_time_mins   8032 non-null   float64       
 4   country                   8032 non-null   object        
 5   favorite_genre            8032 non-null   object        
 6   last_login_date           8032 non-null   datetime64[ns]
 7   customer_support_tickets  8032 non-null   int64         
 8   age_falta                 8032 non-null   int64         
 9   watch_falta               8032 non-null   int64         
 10  genre_falta               8032 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(4)
memory usage: 690.4+ KB


In [ ]:
#PASO 7. RESUMEN FINAL DEL PIPELINE

# 1. Construimos el DataFrame del log acumulado
pipeline_log = pd.DataFrame(log)

print("--- Log completo del pipeline ---")
print(pipeline_log.to_string(index=False))

retencion_final = (len(df) / 8160) * 100

delta_minutos = (minutos_sin_age - minutos_con_age)

print("\n" + "="*30 + " Métricas Finales " + "="*30)
print(f"Retención estructural             : {retencion_final:.1f}%")
print(f"Δ watch_time MNAR (age)           : {delta_minutos:.2f} minutos")
print(f"Outliers minutos recortados (IQR) : {len(outliers_mins)}")
print(f"Outliers tickets recortados (IQR) : {len(outliers_tickets)}")
print("="*78)

# Guardamos el log acumulado en un archivo CSV
pipeline_log.to_csv('pipeline_log.csv', index=False)

--- Log completo del pipeline ---
 Paso                                                      Descripción  Filas  Nulos  Retención (%)
    0                                                 Dataset original   8160    753         100.00
    1                                        Eliminación de duplicados   8032    753          98.43
    2                                         Normalización categorías   8032    759          98.43
    3 Corrección valores imposibles en edad, minutos, tickets y fechas   8032   1406          98.43
    5                   Winsorización robusta IQR en minutos y tickets   8032   1406          98.43
    6                            Imputación diferenciada por mecanismo   8032      0          98.43

============================== Métricas Finales ==============================
Retención estructural             : 98.4%
Δ watch_time MNAR (age)           : -322.48 minutos
Outliers minutos recortados (IQR) : 139
Outliers tickets recortados (IQR) : 81


In [ ]:
# --- EXPORTAR DATASET DEPURADO PARA EL EDA ---

# 1. Guardamos el DataFrame limpio en un archivo CSV físico dentro de Colab
df.to_csv('streaming_users_clean.csv', index=False)

print("¡Archivo 'streaming_users_clean.csv' generado con éxito!")

¡Archivo 'streaming_users_clean.csv' generado con éxito!
